In [1]:
#@title Setup (~13mins)

Run_Boltz = True
Run_Protenix = False

if Run_Protenix:
  !git clone https://github.com/cytokineking/ProtenixScore
  %cd ProtenixScore
  !./install_protenixscore.sh --no-conda
  %cd /content
  !pip install https://data.pyg.org/whl/torch-2.7.0%2Bcu126/torch_cluster-1.6.3%2Bpt27cu126-cp312-cp312-linux_x86_64.whl
if Run_Boltz:
  !pip install virtualenv
  !virtualenv boltz_env
  !source /content/boltz_env/bin/activate; pip install boltz[cuda] -U
  !pip uninstall torch torchvision torchaudio -y
  !pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
  !pip install https://data.pyg.org/whl/torch-2.8.0%2Bcu128/torch_cluster-1.6.3%2Bpt28cu128-cp312-cp312-linux_x86_64.whl

!pip install torch_geometric
!pip install pyrosetta-installer
!python -c 'import pyrosetta_installer; pyrosetta_installer.install_pyrosetta()'
!pip install biopython
!pip install tqdm
!pip install dm-tree
!pip install biotite
!pip install pyyaml
!pip install modelcif
!pip install e3nn
#!pip install easydict


!git clone https://github.com/nedru004/MPNN_affinity_maturation.git
#working_dir = "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation"
working_dir = "/content/MPNN_affinity_maturation"

!git clone https://github.com/dauparas/ProteinMPNN.git

!git clone https://gitlab.com/mjslee0921/flowpacker
#%pip install --use-deprecated=legacy-resolver -r flowpacker/requirements.txt

#!git clone https://github.com/ohuelab/boltzina.git
#!https://github.com/nedru004/boltz_score.git

import sys
sys.path.insert(0, working_dir)
sys.path.insert(0, "/content/flowpacker")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 31.6 MB/s eta 0:00:00
created virtual environment CPython3.12.12.final.0-64-x86_64 in 366ms
  creator CPython3Posix(dest=/content/boltz_env, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.0.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
  Installing build dependencies ... done
  Getting

In [2]:
#@title 1. MPNN Squence Design
import subprocess
import sys
import json # Added for JSON handling
import os   # Added for path operations

mpnn_script_path = f"ProteinMPNN/protein_mpnn_run.py" # Renamed 'script' to avoid conflict
folder_with_pdbs = f"{working_dir}/test/"
path_for_parsed_chains = f"{working_dir}/test/parsed_chains.jsonl"
path_for_assigned_chains = f"{working_dir}/test/assigned_chains.jsonl"
path_for_fixed_positions = f"{working_dir}/test/fixed_positions.jsonl" # Ensuring it is jsonl as per MPNN and code definition
path_for_fixed_positions_csv = f"{working_dir}/test/fixed_positions.csv"
out_dir = f"{working_dir}/test/mpnn_out/"
out_dir_bias = f"{working_dir}/test/mpnn_out_bias/"
bias_AA_file = f"{working_dir}/bias_AA.jsonl"
chains_to_design ="A"
num_seq_per_target=8
sampling_temp =0.5
seed = 37
fixed_positions="11 12 13 14 15" #fixing/not designing residues 1 2 3...25 in chain A and residues 10 11 12...40 in chain C
#generate fixed position csv
with open(path_for_fixed_positions_csv, 'w') as f:
  f.write(fixed_positions)

!python ProteinMPNN/helper_scripts/parse_multiple_chains.py --input_path=$folder_with_pdbs --output_path=$path_for_parsed_chains

!python ProteinMPNN/helper_scripts/assign_fixed_chains.py --input_path=$path_for_parsed_chains --output_path=$path_for_assigned_chains --chain_list "$chains_to_design"

!python ProteinMPNN/helper_scripts/make_fixed_positions_dict.py --input_path=$path_for_parsed_chains --output_path=$path_for_fixed_positions --chain_list "$chains_to_design" --position_list "$fixed_positions"

# Run actual protein_mpnn_run script
args_for_first_mpnn_run = [
    sys.executable, mpnn_script_path,
    "--jsonl_path", path_for_parsed_chains,
    "--chain_id_jsonl", path_for_assigned_chains,
    "--fixed_positions_jsonl", path_for_fixed_positions,
    "--out_folder", out_dir,
    "--num_seq_per_target", str(num_seq_per_target),
    "--sampling_temp", str(sampling_temp),
    "--seed", str(seed),
    "--batch_size", str(num_seq_per_target),
    "--omit_AAs", "C"]

subprocess.run(args_for_first_mpnn_run)


# MPNN with AA bias
# Ensure correct script path and flag for fixed positions.
args_for_second_mpnn_run = [
    sys.executable, mpnn_script_path,
    "--jsonl_path", path_for_parsed_chains,
    "--chain_id_jsonl", path_for_assigned_chains,
    "--fixed_positions_jsonl", path_for_fixed_positions,
    "--out_folder", out_dir_bias,
    "--num_seq_per_target", str(num_seq_per_target),
    "--sampling_temp", str(sampling_temp),
    "--seed", str(seed),
    "--batch_size", str(num_seq_per_target),
    "--omit_AAs", "C",
    "--bias_AA_jsonl", bias_AA_file]

subprocess.run(args_for_second_mpnn_run)

CompletedProcess(args=['/usr/bin/python3', 'ProteinMPNN/protein_mpnn_run.py', '--jsonl_path', '/content/MPNN_affinity_maturation/test/parsed_chains.jsonl', '--chain_id_jsonl', '/content/MPNN_affinity_maturation/test/assigned_chains.jsonl', '--fixed_positions_jsonl', '/content/MPNN_affinity_maturation/test/fixed_positions.jsonl', '--out_folder', '/content/MPNN_affinity_maturation/test/mpnn_out_bias/', '--num_seq_per_target', '8', '--sampling_temp', '0.5', '--seed', '37', '--batch_size', '8', '--omit_AAs', 'C', '--bias_AA_jsonl', '/content/MPNN_affinity_maturation/bias_AA.jsonl'], returncode=0)

In [3]:
#@title 2. Run flowpacker

from utilities import direct_fasta_to_csv

mpnn_folders = [f"{working_dir}/test/mpnn_out/seqs", f"{working_dir}/test/mpnn_out_bias/seqs"]
direct_fasta_to_csv(input_dirs=mpnn_folders, output_csv=f"{working_dir}/test/mpnn_final_result.csv", suffix=".pdb")

# Edit import in flowpacker
file_path = '/content/flowpacker/utils/loader.py'
line_number_to_edit = 45 # Line numbers are 1-based for users
new_content = "    config_dir = '/content/MPNN_affinity_maturation/base.yaml'"
with open(file_path, 'r') as file:
    lines = file.readlines()
# Adjust for 0-based indexing (line 3 is at index 2)
if 0 < line_number_to_edit <= len(lines):
    lines[line_number_to_edit - 1] = new_content + '\n' # Ensure a newline character is present
# Step 3: Write the modified list back to the file
with open(file_path, 'w') as file:
    file.writelines(lines)


# Run flowpacker
!python $working_dir/sampler_pdb_colab.py base --use_gt_masks True --pdb_dir $working_dir/test/ --save_dir $working_dir/test/flowpacker_out/ --csv_file $working_dir/test/mpnn_final_result.csv

✅ Processing complete! 16 unique sequences have been written to: /content/MPNN_affinity_maturation/test/mpnn_final_result.csv
/content/flowpacker/models/equiformer_v2/layer_norm.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/content/flowpacker/models/equiformer_v2/layer_norm.py:152: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/content/flowpacker/models/equiformer_v2/layer_norm.py:227: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/content/flowpacker/models/equiformer_v2/layer_norm.py:310: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp

In [ ]:
#@title 3. Score and filter binders for pTM and ipTM
from MPNN_affinity_maturation.utilities import filter_protenix_scores, filter_boltz_scores, generate_boltz_yamls_from_pdbs

if Run_Protenix:
  # Edit import
  file_path = '/content/ProtenixScore/score.py'
  with open(file_path, 'r') as file:
      lines = file.readlines()
  # Adjust for 0-based indexing (line 3 is at index 2)
  lines[19 - 1] = 'import sys\nsys.path.insert(0, "/content/ProtenixScore/Protenix_fork")\n' # Ensure a newline character is present
  lines[1071 - 1] = '    from ProtenixScore.msa_colabfold import ColabFoldMSAConfig, ensure_msa_dir\n'
  # Step 3: Write the modified list back to the file
  with open(file_path, 'w') as file:
      file.writelines(lines)
  # Edit import
  file_path = '/content/ProtenixScore/cli.py'
  new_content = 'from ProtenixScore.score import run_score'
  with open(file_path, 'r') as file:
      lines = file.readlines()
  lines[5 - 1] = new_content + '\n' # Ensure a newline character is present
  # Step 3: Write the modified list back to the file
  with open(file_path, 'w') as file:
      file.writelines(lines)
  !python ProtenixScore/cli.py score --input $working_dir/test/flowpacker_out/after_pdbs/ --output $working_dir/test/protenix_scores/ --recursive
  # Filter scores
  filter_protenix_scores(f"{working_dir}/test/protenix_scores/", "structure_score_summary.csv", "test/filtered_pdb", threshold_pTM = 0.8, threshold_ipTM = 0.8)

if Run_Boltz:
  # Boltz full prediction
  # write a yaml file for each pdb
  generate_boltz_yamls_from_pdbs(f"{working_dir}/test/flowpacker_out/after_pdbs/", f"{working_dir}/test/boltz_input/")
  for yaml in os.listdir(f"{working_dir}/test/boltz_input/"):
    if yaml.endswith(".yaml"):
      !source /content/boltz_env/bin/activate; boltz predict $working_dir/test/boltz_input/$yaml --out_dir $working_dir/test/boltz_out/ --use_msa_server
  # calculate score and RMSD
  filter_boltz_scores(f"{working_dir}/test/boltz_out/", "structure_score_summary.csv", "test/filtered_pdb", threshold_pTM = 0.8, threshold_ipTM = 0.8)


Generating Boltz YAMLs: 100%|██████████| 8/8 [00:00<00:00, 752.53it/s]


✅ Generated 8 Boltz YAML files in /content/MPNN_affinity_maturation/test/boltz_input/
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Extracting the CCD data to /root/.boltz/mols. This may take a bit of time. You may change the cache directory with the --cache flag.


In [ ]:
#@title 4. Rosetta interaction energy

#pdb_folder = "test"
pdb_file = f"{working_dir}/test/filtered_pdb"
interface_dist = 10
energy_filter = -5

!python $working_dir/per_residue_energy_pyrosetta.py --pdb $pdb_file --binder_id A --target_id M --interface_dist $interface_dist --output_dir $working_dir/test/ --xml_protocol $working_dir/update.xml

In [ ]:
#@title 5. Concatenate key residues

from utilities import process_pdb_mutation_and_renumber
# merge key residues into one pdb file

# Extract Sequences
!python demo_scripts/pdb2fa.py $working_dir/test/af3score/filtered_links csv $working_dir/test/af3score/filtered_links.csv

process_pdb_mutation_and_renumber(f"{working_dir}/test/fixed_residue_pyrosetta.csv", f"{working_dir}/test/merge_motif_pdb/", renumber_chain='M', start_index=9) # The residue numbering of the target chain is consistent with the input target file